In [2]:
from edgar import Company, set_identity
import pandas as pd
from tqdm.auto import tqdm

# REQUIRED: The SEC will block you without a User-Agent string
set_identity("cg77@fordham.edu")

In [9]:
def load_live_filings(tickers, year_range):
    rows = []
    year_min, year_max = year_range

    for ticker in tqdm(tickers, desc="Processing Tickers"):
        try:
            company = Company(ticker)
            filings = company.get_filings(form="10-K")
            if not filings: continue

            for filing in filings:
                f_year = filing.filing_date.year
                if year_min <= f_year <= year_max:
                    
                    # 1. Get the structured TenK object
                    tenk = filing.obj()
                    
                    # 2. Extract Sections (Item 1A, 7, and 8)
                    # We use .get() or dictionary access to avoid AttributeErrors
                    risk_factors = getattr(tenk, "risk_factors", "") or tenk["Item 1A"] or ""
                    mda = getattr(tenk, "management_discussion", "") or tenk["Item 7"] or ""
                    
                    # Item 8 (Financials) is where the Debt Tables live
                    financials = tenk["Item 8"] or ""
                    
                    # Item 3 (Legal)
                    legal = tenk["Item 3"] or ""

                    rows.append({
                        "ticker": ticker.upper(),
                        "year": f_year,
                        "company": filing.company,
                        "section_1a": str(risk_factors),
                        "section_7": str(mda),
                        "section_8": str(financials),
                        "section_3": str(legal)
                    })
        except Exception as e:
            print(f"Skipping {ticker} year {f_year if 'f_year' in locals() else 'unknown'} due to error: {e}")
            continue

    return pd.DataFrame(rows)

In [8]:
# Test with one ticker first
test_df = load_live_filings(["AAPL"], (2023, 2025))
print(f"Successfully loaded {len(test_df)} filings.")
if not test_df.empty:
    display(test_df.head())

Processing Tickers: 100%|██████████| 1/1 [00:37<00:00, 37.45s/it]

Successfully loaded 3 filings.


,ticker,year,company,section_1a,section_7,section_8,section_3
0,AAPL,2025,Apple Inc.,Item 1A. Risk Factors\n\nThe following summ...,Item 7. Management’s Discussion and Analysi...,Item 8. Financial Statements and Supplement...,Item 3. Legal Proceedings\n\nDigital Market...
1,AAPL,2024,Apple Inc.,Item 1A. Risk Factors\n\nThe Company’s busi...,Item 7. Management’s Discussion and Analysi...,Item 8. Financial Statements and Supplement...,Item 3. Legal Proceedings\n\nDigital Market...
2,AAPL,2023,Apple Inc.,Item 1A. Risk Factors\n\nThe Company’s busi...,Item 7. Management’s Discussion and Analysi...,Item 8. Financial Statements and Supplement...,Item 3. Legal Proceedings\n\nEpic Games\n\n...


In [12]:
TICKERS = ["TSLA", "NVDA"]
YEAR_RANGE = (2020, 2025)
TARGET_SECTIONS = ["section_1a", "section_7", "section_3"]

In [13]:
test_df = load_live_filings(TICKERS, YEAR_RANGE )
print(f"Successfully loaded {len(test_df)} filings.")
if not test_df.empty:
    display(test_df.head())

Processing Tickers:   0%|          | 0/2 [00:00<?, ?it/s]TenK falling back to legacy parser for 'Item 1A' (filing: 0001104659-25-042659). New parser sections available: ['part_iii_part_iii', 'part_iii_item_10', 'part_iii_item_11', 'part_iii_item_12', 'part_iii_item_13', 'part_iii_item_14', 'part_iv_part_iv', 'part_iv_item_15']. This fallback will be removed in v6.0.
TenK falling back to legacy parser for 'Item 1A' (filing: 0001104659-25-042659). New parser sections available: ['part_iii_part_iii', 'part_iii_item_10', 'part_iii_item_11', 'part_iii_item_12', 'part_iii_item_13', 'part_iii_item_14', 'part_iv_part_iv', 'part_iv_item_15']. This fallback will be removed in v6.0.
TenK falling back to legacy parser for 'Item 7' (filing: 0001104659-25-042659). New parser sections available: ['part_iii_part_iii', 'part_iii_item_10', 'part_iii_item_11', 'part_iii_item_12', 'part_iii_item_13', 'part_iii_item_14', 'part_iv_part_iv', 'part_iv_item_15']. This fallback will be removed in v6.0.
TenK fal

KeyboardInterrupt: 